In [1]:
import sys
from pathlib import Path
from html import escape

import pandas as pd
from IPython.display import HTML, display

sys.path.append("../")

from evaluation.classes import AnswerRelevanceWrapper
from evaluation.eval import answer_relevance, load_tests
from evaluation.reporting import (
    build_eval_question_report,
    build_retrieved_doc_rows,
    display_eval_question_report,
    export_answer_relevance_report_to_markdown,
)
from pipelines.faq_pipeline import FaqPipeline


/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
async def evaluate_all_answer_relevance(limit=None, markdown_path=None):
    tests = load_tests()
    selected_tests = tests[:limit] if limit is not None else tests
    pipeline = FaqPipeline()
    rows = []
    question_reports = []

    for index, test in enumerate(selected_tests, start=1):
        pipeline_result = pipeline.run(test.question)
        nodes = pipeline_result["nodes"]
        answer = pipeline_result["answer"]

        score = await answer_relevance(
            AnswerRelevanceWrapper(
                question=test.question,
                answer=answer,
            )
        )

        retrieved_docs, doc_rows = build_retrieved_doc_rows(nodes)

        row = {
            "question": test.question,
            "reference_answer": test.reference_answer,
            "generated_answer": answer,
            "relevant_docs": test.relevant_docs,
            "retrieved_docs": retrieved_docs,
            "answer_relevance": score,
        }
        rows.append(row)

        display_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            metrics={"answer_relevance": row["answer_relevance"]},
            text_blocks=[
                ("Reference answer", row["reference_answer"]),
                ("Generated answer", row["generated_answer"]),
            ],
            doc_rows=doc_rows,
            index=index,
            total=len(selected_tests),
        )

        question_report = build_eval_question_report(
            question=test.question,
            relevant_docs=test.relevant_docs,
            retrieved_docs=row["retrieved_docs"],
            doc_rows=doc_rows,
            extra_fields={
                "reference_answer": row["reference_answer"],
                "generated_answer": row["generated_answer"],
                "answer_relevance": row["answer_relevance"],
            },
            index=index,
            total=len(selected_tests),
        )
        question_reports.append(question_report)

    report_df = pd.DataFrame(rows)
    summary_df = report_df[["answer_relevance"]].mean().to_frame(name="mean").T.round(3)

    display(HTML("<h2 style='margin:32px 0 12px;color:#24292f;'>Final metrics</h2>"))
    display(summary_df)

    if markdown_path is not None:
        saved_path = export_answer_relevance_report_to_markdown(
            report_df,
            question_reports,
            Path(markdown_path),
            notebook_path="notebooks/13_faq_answer_relevance_evaluating.ipynb",
        )
        display(HTML(f"<p style='margin:12px 0 0;color:#1a7f37;'>Markdown report saved to: <code>{escape(str(saved_path))}</code></p>"))

    return report_df


In [3]:
await evaluate_all_answer_relevance(markdown_path="../reports/faq_answer_relevance_evaluation.md")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15635.87it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,rank,section_id,title,rerank_score,preview
0,1,93,What to do if the seller does not issue a refund,2.5227,What to do if the seller does not issue a refu...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16450.18it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,rank,section_id,title,rerank_score,preview
0,1,25,One delivery address,3.2498,One delivery address Select one delivery addre...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14109.00it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

,rank,section_id,title,rerank_score,preview
0,1,67,I want to change the delivery day. What should...,5.3827,I want to change the delivery day. What should...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13400.76it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,rank,section_id,title,rerank_score,preview
0,1,74,What you should do if you do not receive your ...,7.3588,What you should do if you do not receive your ...
1,2,77,I have been waiting a long time for my order d...,6.5365,I have been waiting a long time for my order d...


,answer_relevance
mean,0.827


,question,reference_answer,generated_answer,relevant_docs,retrieved_docs,answer_relevance
0,Will I get a full refund including shipping if...,The seller refunds you for the returned produc...,The information was not found. You can call th...,[83],[93],0.165476
1,Can I use multiple delivery addresses for one ...,Select one delivery address and one set of VAT...,You can use only one delivery address for the ...,[25],[25],0.870470
2,Why does my payment status show as canceled or...,Your payment may have the Canceled status if:\...,Your payment may show as canceled if the payme...,[33],"[33, 34]",0.879098
3,How can I find the seller's email and phone nu...,"After completing each transaction, we will ema...","After you make a purchase, you can find the se...",[21],[21],0.994199
4,What should I do if my parcel was damaged when...,If you notice the damage in the pick-up point:...,"If you notice the damage at the pick-up point,...",[69],[69],0.886155
5,How do I sign up using my Google or Facebook a...,Open the sign in page.\nSelect signing in thro...,To sign up using your Google or Facebook accou...,[3],[3],0.982205
6,How can I find my parcel's tracking number and...,You can check the delivery status of your parc...,You can find your parcel's tracking number and...,[42],[42],1.000000
7,How will I receive my refund from the seller?,The refund method depends on the payment metho...,The information was not found in the provided ...,[85],[83],0.285321
8,How long does it take to get a refund for a wi...,You will get the refund within 3 business days...,You will get the refund within 3 business days...,[91],[91],0.592771
9,How is the estimated delivery time calculated ...,For every offer on HopShop and on selected pur...,The estimated delivery time on HopShop is calc...,[43],[43],0.992280
